# Heart Disease Prediction Using Logistic Regression

## THE BIG PICTURE

- The goal: **Predict whether a person is at risk of developing coronary heart disease (CHD) within 10 years.**

We will use information about a person such as:

- age
- gender
- smoking habits
- blood pressure
- cholesterol
- BMI
- heart rate
- glucose level
- etc ...

to predict the value of the **`TenYearCHD`** column.

The target has two possible values:

- `0` → No CHD within 10 years
- `1` → CHD within 10 years

So this is a **classification problem**, not a regression problem.

---

## Simply:

We're going to train a model to answer this question:

> **"Given these characteristics about a person, does the data suggest that they belong to class 0 or class 1?"**

For example:

```text
Person
────────────────────────
Age              → 55
Smoking          → Yes
Blood pressure   → High
Cholesterol      → ...
BMI              → ...
Heart rate       → ...
Glucose          → ...
────────────────────────
             ↓
     Logistic Regression
             ↓
      Prediction = 1

## STEP 1 : Load the dataset

In [2]:
import pandas as pd

df = pd.read_csv("framingham.csv")

print(df.head())

   male  age  education  currentSmoker  cigsPerDay  BPMeds  prevalentStroke  \
0     1   39        4.0              0         0.0     0.0                0   
1     0   46        2.0              0         0.0     0.0                0   
2     1   48        1.0              1        20.0     0.0                0   
3     0   61        3.0              1        30.0     0.0                0   
4     0   46        3.0              1        23.0     0.0                0   

   prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  heartRate  glucose  \
0             0         0    195.0  106.0   70.0  26.97       80.0     77.0   
1             0         0    250.0  121.0   81.0  28.73       95.0     76.0   
2             0         0    245.0  127.5   80.0  25.34       75.0     70.0   
3             1         0    225.0  150.0   95.0  28.58       65.0    103.0   
4             0         0    285.0  130.0   84.0  23.10       85.0     85.0   

   TenYearCHD  
0           0  
1           0  
2 

## STEP 2 : Explore the data 

**1- Check dataframe shape (how many rows/columns) :**

In [3]:
print(df.shape) # (4240, 16) , means the dataframe has 4240 row and 16 columns

(4240, 16)


**2- Check column names and their data types :**

In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
 15  TenYearCHD       4240 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 530.1 KB
None


**3- Look at basic statistics (min, max, mean) for each numeric column :**

In [14]:
df.describe()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
count,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000,4240.000000
mean,0.429245,49.580189,1.979953,0.494104,8.944340,0.029245,0.005896,0.310613,0.025708,236.667689,132.354599,82.897759,25.799005,75.878774,81.600943,0.151887
std,0.495027,8.572942,1.007087,0.500024,11.904777,0.168513,0.076569,0.462799,0.158280,44.328480,22.033300,11.910394,4.070775,12.023937,22.860340,0.358953
min,0.000000,32.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,107.000000,83.500000,48.000000,15.540000,44.000000,40.000000,0.000000
25%,0.000000,42.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,206.000000,117.000000,75.000000,23.077500,68.000000,72.000000,0.000000
50%,0.000000,49.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,234.000000,128.000000,82.000000,25.400000,75.000000,78.000000,0.000000
75%,1.000000,56.000000,3.000000,1.000000,20.000000,0.000000,0.000000,1.000000,0.000000,262.000000,144.000000,90.000000,28.032500,83.000000,85.000000,0.000000
max,1.000000,70.000000,4.000000,1.000000,70.000000,1.000000,1.000000,1.000000,1.000000,696.000000,295.000000,142.500000,56.800000,143.000000,394.000000,1.000000


**4- Check for missing values :**

We can use the <code>isna().sum()</code> method :

* <code>df.isna()</code> goes through every cell in the df (dataframe) and marks the cell: <b>True if empty</b>, <b>False if it has a value</b>

* <code>.sum()</code> counts all the cells marked with True (the empty cells) in each column

In [5]:
print(df.isna().sum())

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64


**5- Check for duplicates :**

In [6]:
# view the duplicated rows :
df[df.duplicated()]

df.duplicated().sum() # 0 , then there are no duplicated rows

np.int64(0)

## STEP 3 : Clean the data

**1- fill missing values with "median" :**

If you're asking why exactly with `median` ? ( I asked that also ! ) , check the first explanation of the "concepts.md" file.

In [7]:
columns = [ 'education','cigsPerDay','BPMeds','totChol','BMI','heartRate','glucose' ] # these are all the column names that have the missing values.

for column in columns:
    df[column] = df[column].fillna(df[column].median())


# let's check if missing values still exist ?

print(df.isna().sum()) #  0 , then there are no missing values.

male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64


**2- Check columns data types :**

* To ensure evrything is ready for the next phase `Data separation (features and target)`

In [8]:
print(df.dtypes)

male                 int64
age                  int64
education          float64
currentSmoker        int64
cigsPerDay         float64
BPMeds             float64
prevalentStroke      int64
prevalentHyp         int64
diabetes             int64
totChol            float64
sysBP              float64
diaBP              float64
BMI                float64
heartRate          float64
glucose            float64
TenYearCHD           int64
dtype: object


**IMPORTANT RECALL :**

Question : why every value in the data must be a number and every column data type must be int or float ?

* Because : **Most ML models need numerical inputs as they perform mathematical calculations (like `Sigmoid` `Normal Equation` ... etc) on the features.**

## STEP 4 : Separate features and target

- `x` is the entire dataset without the target column (the input data)

- `y` is the target data (the correct output, which is **TenYearCHD**)

In [9]:
x = df.drop(columns=["TenYearCHD"])

y = df["TenYearCHD"]

## Step 5: Split into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.25 , random_state=42)

**How does the `train_test_split()` method works ?**

* We **give the function the features** (the person's charachteristics) **"x"** **and the target** (TenYearCHD) **"y"**

* **Then , the method shuffles (mix) the rows randomly, then splits the rows into two groups** :

    - **Training group (X_train and y_train)** : the bigger chunk (75% if test_size=0.25). This is what the model learns from.

    - **Testing group (X_test and y_test)** : the smaller chunk (25%). This is used only after training, to check how well the model performs on data it never saw during training.

* **test_size=0.25** means 25% of the rows go to testing (4241 × 0.25 = 1060 row for testing), 75% go to training (4241 × 0.75 = 3180 row for training)

* **random_state=42** is used to ensure fair and reproducible experiments. to ensure that the train/test split stays the same every time we change something to the model, which makes our experiments reproducible and easier to compare.(If you need more explanations check the "concepts" file - question 2)


## Step 6: Feature Scaling


Get a Recall about "Logistic Regression" from the "LogisticRegression_Explained" file, to recall how "logistic Regression" works first , then continue Feature Scaling explanation.

**Introduction :**

* In machine learning , the difference between a high-performing model and a struggling-model are small details.

* One of the most overlooked steps in the training process is Feature scaling , so Feature scaling is an important step.

* Feature scaling may seem minor, but how you scale data can affect model accuracy, training speed, and stability.

* Not every scaling method works well for every situation , Because :

    - A good scaling method can: improve model performance , Keep features balanced.

    - A bad scaling method choice can : Distort relationships between features , hurt model performance.

<h3><b>What is Feature Scaling ?</b></h3>

* **Feature Scaling** is **a data preprocessing technique** used in machine learning **to normalize or standardize** **the range of** independent variables (**features**)

* Features can have different units and numbers , ex : Age[20-100], Income[20000-100000] , So without scaling : a feature with larger numbers can have too much influence on the model

* Feature scaling is important for models that rely on Distances and Gradient calculations

* Feature Scaling helps make the features more balanced and ensures all features contribute proportionally to the model.

**When do we need Feature Scaling ?**

Simply :

* We need Feature scaling when our features have very different numerical ranges like Age[18-80] and Income[1000-10000] , especially for models that rely on distance or gradient calculations.

Examples: Logistic Regression, Linear Regression, KNN, K-Means, SVM, Neural Networks.

<h3><b>Why Feature Scaling matters ?</b></h3>

* **Improves model performance** : Algorithms like gradient descent converge faster when features are normalized, since they don’t have to “zig-zag” across uneven scales.

* **Interpretability** : Standardized features (mean 0, variance 1) make it easier to compare the relative importance of coefficients in linear models.

* **Better Accuracy** : Distance-based models such as k-nearest neighbors (KNN), k-means, and support vector machines (SVMs) perform more reliably with scaled features.

* **Faster Convergence** : Neural networks and gradient descent optimizers reach optimal solutions more quickly when features are scaled.

<h2><b>Common Feature Scaling Techniques</b></h2>

<h4><b>1. Normalization :</b></h4>

* Normalization is one of the simplest and most widely used feature scaling techniques.

* Normalization changes feature values to a fixed range, the most common range is [0-1]

* We can use another custom range [a,b]

* Formula : 

    - `X scaled ​= Xᵢ​ / ∥X∥`​  : 

    - **Xᵢ** -> is the individual feature value you want to scale , if a patient's age is 50, then Xᵢ = 50

    - **∥X∥** -> is the magnitude/length of the feature vector *X* , where :  **∥X∥** = $\sqrt{x1^2 + x2^2 + x3^2 + ... + xn^2}$

    - **X scaled** -> the new scaled value after normalization.
​

## Step 7 : choose and train the model